In [ ]:
#Import tools and set up environment
import os
from dotenv import load_dotenv

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import gradio as gr

# Load environment variables (make sure you have a .env file with OPENAI_API_KEY)
load_dotenv()

#Load Nestlé HR policy and split text
pdf_path = "nestle_hr_policy.pdf"  
loader = PyPDFLoader(pdf_path)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

#Create vector embeddings using Chroma DB and OpenAI
embedding_model = OpenAIEmbeddings()
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_store"  # folder to persist vector DB
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

#Define prompt template to guide the assistant
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI assistant that answers questions about Nestlé's HR policies.
Use only the provided context to respond.
If the answer is not in the context, say "I couldn't find that information."

Context: {context}
Question: {question}
Answer:"""
)

#Build the question-answering system using GPT-3.5 Turbo
llm = ChatOpenAI(model_name="gpt-3.5-turbo")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

#Create Gradio chatbot interface
def chat_with_nestle_bot(user_query):
    try:
        result = qa_chain({"query": user_query})
        return result["result"]
    except Exception as e:
        return f"Error: {str(e)}"

interface = gr.Interface(
    fn=chat_with_nestle_bot,
    inputs=gr.Textbox(label="Ask Nestlé HR Assistant"),
    outputs=gr.Textbox(label="Response"),
    title="Nestlé HR Policy Assistant",
    description="Ask questions about Nestlés HR policies (e.g., leave, benefits, remote work)."
)

#Launch the Gradio app
interface.launch()


Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------


In [ ]:
from langchain.prompts import PromptTemplate

# Define the prompt that will be used for the RetrievalQA chain
prompt_template = PromptTemplate.from_template("""
You are an HR assistant for Nestlé. Use only the provided context to answer the user's question.
If the answer is not available in the context, respond with "I'm sorry, I could not find that information."

Context:
{context}

Question:
{question}
""")

from langchain.chains import RetrievalQA
from langchain.chains.combine_documents import StuffDocumentsChain
from langchain.chains.llm import LLMChain

# LLM
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Wrap prompt into a chain
question_chain = LLMChain(llm=llm, prompt=prompt_template)
document_chain = StuffDocumentsChain(llm_chain=question_chain)

# QA Chain with template applied
qa_chain = RetrievalQA(
    retriever=retriever,
    combine_documents_chain=document_chain,
    return_source_documents=True
)
